# [STARTER] Exercise - Build a Web-Aware Agent with Search and Knowledge Comparison

In this exercise, you'll build an agent that can search the web for current information and compare
it with its internal knowledge. This demonstrates how to enhance an LLM's capabilities with real-time
web data and how to critically analyze differences between sources.


## Challenge

Your task is to create an agent that can:

- Implement web search functionality using Tavily API
- Parse and process search results effectively
- Handle different types of queries (news, facts, events)
- Extract relevant information from search results

## Setup
First, let's import the necessary libraries:

In [1]:
import os
from datetime import datetime
from typing import List, Dict
from dotenv import load_dotenv
from tavily import TavilyClient

from lib.agents import Agent
from lib.messages import BaseMessage
from lib.tooling import tool

In [2]:
load_dotenv()

True

## Play with Tavily

In [3]:
api_key = os.getenv("TAVILY_API_KEY")
client = TavilyClient(api_key=api_key)

In [4]:
# [TODO] Define a query and run
query = "What's Nintendo?"
result = client.search(query)

In [5]:
result

{'query': "What's Nintendo?",
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.nintendo.com/au/about/?srsltid=AfmBOoqtTJaDLOEqxKSy1-Et-lnLBpJSxe2h1gWqEgjHII3EPt-4M3tu',
   'title': 'About Nintendo - Official Site - Video Game Consoles, Games',
   'content': 'Nintendo is an international leader in the interactive entertainment industry, and develops, produces and markets software and hardware.',
   'score': 0.6530611,
   'raw_content': None},
  {'url': 'https://en.wikipedia.org/wiki/Nintendo',
   'title': 'Nintendo - Wikipedia',
   'content': '* [Afrikaans](https://af.wikipedia.org/wiki/Nintendo "Nintendo – Afrikaans") * [Alemannisch](https://als.wikipedia.org/wiki/Nintendo "Nintendo – Alemannic") * [Aragonés](https://an.wikipedia.org/wiki/Nintendo "Nintendo – Aragonese") * [Arpetan](https://frp.wikipedia.org/wiki/Nintendo "Nintendo – Arpitan") * [Azərbaycanca](https://az.wikipedia.org/wiki/Nintendo "Nintendo – Azerbaijani") * [Беларуская](

## Define Web Search tool

In [9]:
# [TODO] Define the search tool
@tool
def web_search(query: str, search_depth: str = "advanced") -> Dict:
    """
    Search the web using Tavily API
    args:
        query (str): Search query
        search_depth (str): Type of search - 'basic' or 'advanced' (default: advanced)
    """
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)
    
    # Perform the search
    search_result = client.search(
        query=query,
        search_depth=search_depth,
        include_answer=True,
        include_raw_content=False,
        include_images=False
    )
    
    # Format the results
    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": query
        }
    }
    
    return formatted_results

In [10]:
tools = [web_search]

## Define Agents

The first agent should not use tools, just its own knowledge. The second one should have web search tool enabled.

In [11]:
# [TODO] Agent with no tools
simple_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=("You are a helpful assistant"),
)

In [12]:
# [TODO] Agent with web search tool
web_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
            "You are a web-aware assistant that can search for update information "
            "For each query, you will search the web for current information using " 
            "Tavily's AI-optimized search and provide a comprehensive answer\n"
            "Always cite your sources and explain any discrepancies found.\n"
            "Be particularly attentive to dates and time-sensitive information."
    ),
    tools=tools
)

In [13]:
def print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

## Run your Agents

Run the Agents and compare them. The following queries are just for reference. Change them as you want.

**Simple Agent**

**Note**: This example relies on the date being recent enough that the answer will not be in the model's training data. Try with other current events/dates if needed to get similar results.

In [14]:
run1 = simple_agent.invoke(
    query="Who won the 2025 Oscar for International Movie?", 
)

print("\nMessages from run 1:")
messages = run1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are a helpful assistant, tool_calls = None)
 -> (role = user, content = Who won the 2025 Oscar for International Movie?, tool_calls = None)
 -> (role = assistant, content = I'm sorry, but I don't have information on events that occurred after October 2023. To find out the winner of the 2025 Oscar for International Movie, I recommend checking the latest news or the official Oscars website., tool_calls = None)


In [15]:
print(run1.get_final_state()["messages"][-1].content)

I'm sorry, but I don't have information on events that occurred after October 2023. To find out the winner of the 2025 Oscar for International Movie, I recommend checking the latest news or the official Oscars website.


In [16]:
run2 = simple_agent.invoke(
    query="What are the most recent developments in AI technology?", 
)
print("\nMessages from run 2:")
messages = run2.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 2:
 -> (role = system, content = You are a helpful assistant, tool_calls = None)
 -> (role = user, content = Who won the 2025 Oscar for International Movie?, tool_calls = None)
 -> (role = assistant, content = I'm sorry, but I don't have information on events that occurred after October 2023. To find out the winner of the 2025 Oscar for International Movie, I recommend checking the latest news or the official Oscars website., tool_calls = None)
 -> (role = user, content = What are the most recent developments in AI technology?, tool_calls = None)
 -> (role = assistant, content = As of my last knowledge update in October 2023, here are some of the notable developments in AI technology up to that point:

1. **Generative AI Advances**: Tools like GPT-4 and other generative models have continued to evolve,

In [17]:
print(run2.get_final_state()["messages"][-1].content)

As of my last knowledge update in October 2023, here are some of the notable developments in AI technology up to that point:

1. **Generative AI Advances**: Tools like GPT-4 and other generative models have continued to evolve, enabling more sophisticated text, image, and even audio generation. New models are being developed with improved capabilities in understanding context and producing coherent and contextually relevant content.

2. **AI in Healthcare**: AI applications in healthcare have expanded, with advancements in diagnostics, personalized medicine, and drug discovery. AI algorithms are increasingly being used to analyze medical images, predict patient outcomes, and assist in treatment planning.

3. **AI Ethics and Regulation**: There has been a growing emphasis on ethical considerations in AI development. Discussions around regulation, transparency, and accountability have intensified, leading to proposals for frameworks to govern AI usage and mitigate risks.

4. **AI for Cli

**Web Agent**

In [18]:
run1 = web_agent.invoke(
    query="Who won the 2025 Oscar for International Movie?", 
)

print("\nMessages from run 1:")
messages = run1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are a web-aware assistant that can search for update information For each query, you will search the web for current information using Tavily's AI-optimized search and provide a comprehensive answer
Always cite your sources and explain any discrepancies found.
Be particularly attentive to dates and time-sensitive information., tool_calls = None)
 -> (role = user, content = Who won the 2025 Oscar for International Movie?, tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageFunctionToolCall(id='call_7sb2F004mO4eOwzPT0kyN5Gj', function=Function(arguments='{"query":"2025 Oscar winner International Movie","search_depth":"advanced"}', name='we

In [19]:
print(run1.get_final_state()["messages"][-1].content)

At the 2025 Academy Awards, the Oscar for Best International Feature Film was awarded to **"I'm Still Here"**, a Brazilian film directed by Walter Salles. This marked Brazil's first Oscar win in this category. The film tells the poignant story of a mother and activist dealing with the forced disappearance of her husband during Brazil's military dictatorship in the 1970s. The film is based on the biographical book by Marcelo Rubens Paiva, who is also the real-life subject of the story.

In addition to "I'm Still Here," other nominees for the Best International Feature included "The Girl with the Needle" (Denmark), "Emilia Pérez" (France), "The Seed of the Sacred Fig" (Germany), and "Flow" (Latvia) (NPR, Hollywood Reporter, People).

For more details about the ceremony and other winners, you can refer to the [Hollywood Reporter](https://www.hollywoodreporter.com/lists/oscars-2025-winners-list/) or [NPR](https://www.npr.org/2025/03/02/nx-s1-5315313/oscars-2025-im-still-here-brazil-best-in

In [20]:
run2 = web_agent.invoke(
    query="What are the most recent developments in AI technology?", 
)
print("\nMessages from run 2:")
messages = run2.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 2:
 -> (role = system, content = You are a web-aware assistant that can search for update information For each query, you will search the web for current information using Tavily's AI-optimized search and provide a comprehensive answer
Always cite your sources and explain any discrepancies found.
Be particularly attentive to dates and time-sensitive information., tool_calls = None)
 -> (role = user, content = Who won the 2025 Oscar for International Movie?, tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageFunctionToolCall(id='call_7sb2F004mO4eOwzPT0kyN5Gj', function=Function(arguments='{"query":"2025 Oscar winner International Movie","search_depth":"advanced"}', name='we

In [21]:
print(run2.get_final_state()["messages"][-1].content)

As of 2025, several significant developments in artificial intelligence (AI) technology have emerged, reshaping various sectors and increasing AI's influence globally. Here are some of the key trends and advancements:

1. **Rise of Open-Source AI**: China has taken a prominent lead in open-source AI technologies. The release of the Deepseek R1 model marked a turning point, making it the second-ranking AI model globally and demonstrating the capability to be developed at a fraction of the cost of its Western counterparts. This shift has raised concerns in the U.S. regarding AI competitiveness (Time Magazine).

2. **Advancements in Reasoning Models**: AI models have become more capable, particularly with the introduction of "reasoning models." These models can generate extensive reasoning chains, leading to improved answers for complex questions. In 2025, such models achieved notable successes, including winning gold medals at the International Math Olympiad (Time Magazine).

3. **Integr

## Advanced

You can modify `agents.py` to include: 
- a comparison field in the state schema
- a web search step
- a comparison step in the workflow